# Benchmark de manipulação: falsos positivos e negativos, prompts e alinhamento geométrico

Este notebook executa o plano de [report/analise_fp_fn_manipulacao.md](../report/analise_fp_fn_manipulacao.md)
sobre o dataset de manipulação do ViDDAR: 114 pares (imagem crua I_r, imagem aumentada I_a), sendo 44
ataques (M = 1) e 70 benignos (M = 0). Cada par tem o rótulo dos três fatores do artigo: **A**
(alinhamento ao objeto real), **S** (estilo semelhante ao mundo real) e **I** (deturpação da
informação do objeto). Há ataque quando os três são verdadeiros: M = A ∧ S ∧ I (Eq. 7).

A pergunta é **por que o detector erra**, nos falsos positivos (FP) e nos falsos negativos (FN), e se
ajustes de prompt (caminho A) ou um componente geométrico (caminho C2) reduzem esses erros. As
variantes não são propostas de solução: cada uma testa uma hipótese sobre a origem dos erros, e o
conjunto serve para caracterizar o problema em cada VLM.

## Roteiro

| Seção | O que mostra |
|---|---|
| 2 | configuração e texto exato de cada prompt |
| 3 | execução das variantes (só roda o que ainda não existe em `results/`) |
| **4** | **resultado geral: acurácia, precisão e recall de cada variante e se houve ganho em relação ao artigo** |
| 5 | análise dos erros: causas dos FP e FN, respostas por fator e oráculo |
| 6 | caminho A: comparação visual das variantes de prompt |
| **7** | **caminho C2: o A geométrico melhorou a decisão?** |
| 8 | resumo |

## O pipeline anterior (referência)

É o do artigo (§5.2.1), já executado com `scripts/run_manipulation.py --prompt paper`:

1. Uma chamada ao VLM por par, com as duas imagens (crua, depois aumentada) e o prompt de 6 perguntas.
2. As respostas 3, 4 e 5 são lidas como os fatores A, S e I (primeiro "yes"/"no" de cada resposta).
3. A decisão `pred` é o último "yes"/"no" da resposta, ou seja, a pergunta 6.

## O que este notebook acrescenta

| Mudança | Caminho | Chama o VLM? |
|---|---|---|
| Acurácia, precisão e recall (como na Tabela 2 do artigo), FPR, FNR e teste de McNemar em todas as execuções | análise | não |
| Atribuição de cada erro a uma causa e oráculo contrafactual | análise | não |
| Prompts `neutral` e `definition` (6 perguntas, uma chamada) | A | sim |
| Modo decomposto: uma chamada por fator, decisão em código | A | sim, 4 chamadas por par |
| Fator A medido em pixels (Grounding DINO + SAM) e decisão híbrida | C2 | não, só Grounding DINO + SAM |

**O que não muda:** dataset, decodificação greedy (temperatura 0), resolução das imagens, limite de
768 tokens, ordem das imagens e a regra de decisão do artigo no modo conjunto.

## Modelos

Qwen3-VL-2B, Qwen3-VL-4B e LLaVA-Next-8B (4 bits). Para incluir outro VLM, acrescente o preset em
`MODELS` (seção 2): todas as tabelas e gráficos se ajustam sozinhos.

## Convenções

- **Acurácia** = acertos / 114. **Precisão** = ataques reais entre os pares marcados como ataque.
  **Recall** = ataques detectados / 44 (= 1 − FNR). **FPR** = benignos marcados como ataque / 70.
- Resposta ilegível conta como erro, nunca é descartada.
- **Houve ganho?** Uma variante melhora quando a acurácia sobe e o teste de McNemar (pareado, mesmos
  114 pares) dá p < 0,05. Com 114 pares, diferenças de poucos pontos não são significativas.
- Um detector que nunca dispara já tem 61,4% de acurácia (70/114). Por isso olhe também o recall e o
  FNR: uma acurácia alta com recall baixo significa que o detector quase não dispara.

## Antes de começar

- Use o kernel **"Python (ViDDAR .venv)"** e rode as células em ordem.
- A lógica fica em `viddar/`; o notebook só chama as funções. Os mesmos experimentos rodam pela linha
  de comando: `scripts/run_manipulation.py` e `scripts/run_alignment.py`.
- As respostas do VLM ficam em cache (`results/cache/`); execuções prontas só são relidas.

## 1. Ambiente

In [ ]:
import os
import sys
from pathlib import Path

# O notebook pode ser aberto a partir da raiz ou da pasta notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "viddar").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from viddar import manipulation, prompts, reporting, viz
from viddar.experiments import AlignmentConfig, ManipulationConfig, manipulation_variant, run_alignment, run_manipulation

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)
print("raiz do repositório:", ROOT)

## 2. Configuração

- **`MODELS`:** presets de `viddar/vlm.py` a avaliar.
- **`VARIANTS`:** pares (prompt, modo). `joint` é o modo do artigo (uma chamada, 6 perguntas);
  `decomposed` faz uma chamada por fator. A pasta de saída é o nome do prompt, mais `_decomposed` no
  segundo modo. Mantenha `("paper", "decomposed")` antes das outras variantes decompostas: elas
  reaproveitam do cache as chamadas iguais.
- **`REFERENCE`:** variante contra a qual se mede o ganho (o prompt do artigo).
- **`RUN_MISSING`:** com `True`, executa o que ainda não existe; com `False`, só lê `results/`.
- **`RUN_ALIGNMENT`:** executa o caminho C2 (seção 7) nas execuções que ainda não o têm.
- **`LIMIT`:** número de pares, para testes rápidos. Grava em `results/_smoke`; apague a pasta depois.

In [ ]:
MODELS = ["qwen3-vl-2b", "qwen3-vl-4b", "llava-next-8b-4bit"]
VARIANTS = [("paper", "joint"), ("neutral", "joint"), ("definition", "joint"),
            ("paper", "decomposed"), ("definition", "decomposed")]
REFERENCE = "paper"               # prompt do artigo: base de comparação

RUN_MISSING = True        # executa variantes que ainda não existem
RUN_ALIGNMENT = True      # executa o caminho C2 nas execuções que ainda não o têm
LIMIT = None              # None = dataset completo; ex.: 5 para teste rápido

MANIFEST = "data/manifests/viddar_manipulation.csv"
OUT_ROOT = Path("results" if LIMIT is None else "results/_smoke")
VARIANT_NAMES = [manipulation_variant(p, m) for p, m in VARIANTS]
print("resultados em:", OUT_ROOT, "| variantes:", VARIANT_NAMES)

### 2.1 Texto dos prompts

A célula abaixo mostra o texto exato enviado ao VLM. Em resumo, o que muda em relação ao `paper`:

- **`neutral`:** "the second one is an augmented image, ~~created by adding some virtual content to the
  space~~ **which may contain virtual content added to the space**". Na pergunta 5, sai *"Think
  creatively, only say no if the combination has no specific semantic relation."*
- **`definition`:** a pergunta 5 começa pela definição de I do §3.2 do artigo (*"Information
  misrepresentation means that the combination of virtual content and the real world causes users to
  misunderstand the scene: they might overlook critical details or misinterpret non-existent
  information as real."*), mais uma frase nossa: *"Consider a user who takes the virtual content as part
  of the real world."*
- **Modo decomposto (`*_decomposed`):**
  - **chamada 1:** as perguntas 1 e 2 do artigo (conteúdo virtual e objeto-chave), até 4 palavras cada;
  - **chamadas 2 a 4:** cada uma cita o conteúdo e o objeto da chamada 1 e repete a pergunta 3, 4 ou 5 do
    artigo, pedindo que a resposta comece por "yes" ou "no";
  - não há pergunta 6: M = A ∧ S ∧ I é calculado em código. Um fator ilegível conta como erro.
- **`definition_decomposed`:** igual ao `paper_decomposed`, com a definição de I na chamada do fator I.

In [ ]:
def show_prompt(title: str, text: str) -> None:
    display(Markdown(f"**{title}**"))
    print(text, end="\n\n")


for name in ["paper", "neutral", "definition"]:
    show_prompt(f"{name} (modo conjunto, 1 chamada por par)", prompts.MANIPULATION[name])
for call, text in prompts.MANIPULATION_DECOMPOSED["paper"].items():
    show_prompt(f"paper_decomposed, chamada {call}", text)
show_prompt("definition_decomposed, chamada I (as demais são iguais às de paper_decomposed)",
            prompts.MANIPULATION_DECOMPOSED["definition"]["I"])

## 3. Execução das variantes

Para cada modelo e variante que ainda não existe em disco:

- **modo conjunto:** uma chamada por par. As respostas 3–5 viram `pred_A`, `pred_S` e `pred_I`; a
  resposta 2 vira `key_object` e a frase curta `key_phrase` (usada no C2). `pred` é a decisão do artigo;
- **modo decomposto:** quatro chamadas por par, com as respostas brutas em `resp_objects`, `resp_A`,
  `resp_S` e `resp_I`. `pred` é A ∧ S ∧ I.

Cada execução grava `results/manipulation/<modelo>/<variante>/{predictions.csv, metrics.json}` com as
respostas brutas, e tudo depois desta célula é recalculado sem chamar o modelo.

In [ ]:
for model in MODELS:
    for prompt, mode in VARIANTS:
        run_dir = OUT_ROOT / "manipulation" / model / manipulation_variant(prompt, mode)
        if (run_dir / "metrics.json").exists():
            continue
        if not RUN_MISSING:
            print("falta:", run_dir)
            continue
        print("executando:", run_dir)
        run_manipulation(ManipulationConfig(manifest=MANIFEST, vlm=model, prompt=prompt, mode=mode, limit=LIMIT,
                                            out=str(OUT_ROOT / "manipulation")))

runs = reporting.load_manipulation_runs(OUT_ROOT, MODELS, VARIANT_NAMES)
print(f"{len(runs)} execuções carregadas:", sorted(f"{m}/{v}" for m, v in runs))

## 4. Resultado geral: houve ganho?

Uma linha por modelo e variante, com as métricas da Tabela 2 do artigo (acurácia, precisão e recall)
e as taxas de erro (FPR e FNR), todas em %.

- **Δ acurácia:** diferença, em pontos percentuais, para o prompt do artigo (`paper`) no mesmo modelo.
- **McNemar p:** o teste compara as duas decisões nos mesmos 114 pares, usando só os pares em que uma
  acerta e a outra erra.
- **ganho?:** "melhorou" ou "piorou" quando a acurácia muda com p < 0,05; senão, "sem diferença
  significativa".

Para entender **o que** mudou, compare precisão e recall: se o recall cai muito, a variante ganhou
acurácia passando a disparar menos (menos FP, mais FN).

In [ ]:
gain = reporting.gain_table(runs, REFERENCE)
display(gain)
display(Markdown("**Referência: Tabela 2 do artigo (prompt `paper`)**"))
display(reporting.paper_reference())

## 5. Análise dos erros

Esta seção não chama nenhum modelo: tudo é recalculado a partir de `predictions.csv`. Ela segue a
Seção 2 de `report/analise_fp_fn_manipulacao.md` e vale para qualquer variante. O ponto de partida é a
variante `paper`; as mesmas tabelas mostram, para as outras variantes, qual causa de erro mudou.

### 5.1 Taxas de erro com intervalo de confiança

Contagens e intervalos de Wilson (95%). `só variante acerta` / `só paper acerta` são os pares
discordantes do teste de McNemar.

In [ ]:
errors = reporting.error_comparison(runs, REFERENCE)
show_cols = ["model", "variant", "Acurácia % (k/n)", "Precisão % (k/n)", "Recall % (k/n)", "FPR % (k/n)",
             "FPR IC95", "FNR % (k/n)", "FNR IC95", "só variante acerta", f"só {REFERENCE} acerta", "McNemar p"]
display(errors[[c for c in show_cols if c in errors.columns]])

### 5.2 Causas dos FP e dos FN

Cada erro recebe uma causa, nesta ordem:

1. **ilegível:** a decisão ou alguma resposta sobre os fatores não pôde ser lida;
2. **conclusão:** as respostas 3–5 levam a uma decisão, e a pergunta 6 diz o contrário (só existe no
   modo conjunto);
3. **errou X (percepção):** a conclusão segue as respostas, e o erro está no fator X.
   - Num **FP**, X é falso (X = 0) e o modelo disse "sim".
   - Num **FN**, todos os fatores são verdadeiros (é um ataque) e o modelo disse "não" a X.

Cada célula traz o número de erros sobre o número de pares em que aquela causa pode ocorrer. Por
exemplo, um FP "errou A" só pode ocorrer nos 32 benignos cujo único fator falso é A.

In [ ]:
causes = reporting.cause_comparison(runs)
for model in MODELS:
    cols = ["erro", "causa", "pares possíveis"] + [c for c in causes.columns if c.startswith(model + " /")]
    if len(cols) > 3:
        display(Markdown(f"**{model}**"))
        display(causes[cols])

### 5.3 Visão geral: todos os erros (FP + FN) por causa

Cada célula traz o total de erros com aquela causa e quantos são FP e FN; a última linha soma todos os
erros da execução. É a forma mais rápida de ver a causa dominante de cada modelo e como cada variante
a altera.

In [ ]:
overview = reporting.cause_overview(runs)
for model in MODELS:
    cols = ["causa"] + [c for c in overview.columns if c.startswith(model + " /")]
    if len(cols) > 1:
        display(Markdown(f"**{model}**"))
        display(overview[cols])

### 5.4 Respostas por fator

Para cada fator, com que frequência o modelo o classificou errado:

- **errou com X = 0:** respondeu "sim" quando o fator é falso. Alimenta os FP.
- **errou com X = 1:** respondeu "não" quando o fator é verdadeiro. Alimenta os FN.

Um modelo que discrimina bem tem as duas colunas baixas. Uma coluna perto de 100% e a outra perto de 0%
indica que o modelo dá sempre a mesma resposta àquele fator, sem olhar a imagem.

In [ ]:
display(reporting.factor_comparison(runs))

### 5.5 Oráculo contrafactual

**O que é.** O oráculo responde à pergunta: *"e se o modelo acertasse o fator X, mantendo as outras
respostas?"*. Trocamos a resposta do modelo sobre X pelo rótulo verdadeiro, mantemos as outras duas e
recalculamos M = A ∧ S ∧ I. Nenhum modelo é consultado de novo.

**Para que serve.** A diferença entre `oráculo none` e `oráculo X` é o **ganho máximo** de uma
intervenção que corrija só o fator X. O oráculo em A, por exemplo, é o teto do caminho C2 (seção 7), que
só melhora A. Se corrigir X não muda o resultado, não adianta investir em X enquanto os outros fatores
continuarem errados.

**Como ler.** Cada célula é FPR / FNR, em %. A coluna `oráculo none` usa A ∧ S ∧ I das respostas, não
a pergunta 6; no modo conjunto ela pode diferir da decisão `pred`.

**O que se viu com o prompt do artigo** (detalhes na Seção 2.4 do relatório):

- **Qwen3-VL-4B:** com I correto, a FNR cairia de 93% para 7%, mas a FPR subiria de 4% para 47%. Quase
  todos os ataques perdidos vêm da leitura de I, e a rejeição de I escondia erros em A e S.
- **LLaVA:** com A correto, a FPR cairia de 90% para 43%, a maior queda de um fator isolado; só com A e
  S corretos chegaria a 16% / 0%.
- Nenhum fator corrigido sozinho resolve: os modelos erram mais de um julgamento.

In [ ]:
display(reporting.oracle_comparison(runs))

### 5.6 Interpretação por modelo (prompt do artigo)

**Qwen3-VL-2B: respostas constantes e conclusão desconectada.**
- Diz "sim" a A e S em praticamente todos os pares e "não" a I em todos.
- Na pergunta 6 conclui "sim" mesmo depois de ter respondido "não" na 5: todos os 68 FP são erros de
  conclusão (conferido nas respostas brutas, não é erro de parsing).
- Corrigir só a conclusão não resolve: as respostas por fator são constantes, e o modelo passaria a
  nunca detectar. O problema é de percepção e de instrução ao mesmo tempo.

**Qwen3-VL-4B: leitura restritiva de I.**
- 38 dos 41 FN vêm de errar I: o modelo trata adições plausíveis como inofensivas (*"It is a plausible
  accessory that enhances the scooter's utility"*). Para o ViDDAR é o contrário: ser plausível é o que
  engana o usuário.
- A FPR baixa (4%) não é mérito: vem de rejeitar I. Com I correto (oráculo), a FPR sobe para 47%.

**LLaVA-Next-8B: viés de "sim".**
- Todos os FP são de percepção: errou A em 31 dos 32 desalinhados e S em 22 dos 25 com textura
  artificial. Aceita I em 100% dos pares.
- O modelo às vezes percebe o desalinhamento e ainda assim aceita A: descreve *"a cup of coffee ...
  floating above the fan"* e responde que o conteúdo está alinhado.

Para ver o que cada variante mudou, compare as colunas das tabelas 5.2 e 5.3.

### 5.7 Exemplos de erros

Escolha um modelo, uma variante e uma causa para ver o par e a resposta completa do VLM. Os códigos das
causas são `FP:accepted:<X>` (FP "errou X"), `FN:rejected:<X>` (FN "errou X"), `FP:conclusion`,
`FN:conclusion` e `*:illegible`.

In [ ]:
EX_MODEL, EX_VARIANT, EX_CAUSE, N_EXAMPLES = "llava-next-8b-4bit", "paper", "FP:accepted:A", 1

key = (EX_MODEL, EX_VARIANT)
if key in runs:
    df = runs[key].assign(cause=manipulation.error_causes(runs[key]))
    df = df.merge(pd.read_csv(MANIFEST, dtype={"id": str})[["id", "raw_path", "ar_path"]], on="id")
    df[["raw_path", "ar_path"]] = df[["raw_path", "ar_path"]].map(lambda p: str((Path(MANIFEST).parent / p).resolve()))
    sample = df[df["cause"] == EX_CAUSE].head(N_EXAMPLES)
    print(f"{(df['cause'] == EX_CAUSE).sum()} pares com causa {EX_CAUSE}")
    for _, row in sample.iterrows():
        viz.show_manipulation(row)
        plt.show()

## 6. Caminho A: comparação visual das variantes

Um gráfico por modelo. Cada ponto é uma variante. O canto (0, 0) é o ideal, e a diagonal tracejada é
um classificador que ignora a entrada. Uma variante melhora de fato quando o ponto se aproxima do canto;
se ele só desliza ao longo da diagonal, a variante trocou FP por FN. A tabela ao lado traz os mesmos
números.

In [ ]:
for model in MODELS:
    t = errors[errors["model"] == model][["variant", "fpr", "fnr"]]
    if t.empty:
        continue
    fig, table = viz.operating_points(t, f"{model}: FPR × FNR por variante")
    plt.show()
    display(table)

**Melhor variante por modelo:** a de maior acurácia, como no artigo. Ela alimenta o caminho C2 (seção
7), junto com o prompt do artigo.

In [ ]:
best = errors.loc[errors.groupby("model")["accuracy"].idxmax(), ["model", "variant", "accuracy", "precision",
                                                                   "recall"]]
display((best.set_index(["model", "variant"]) * 100).round(1)
        .rename(columns={"accuracy": "acurácia (%)", "precision": "precisão (%)", "recall": "recall (%)"}))
BEST = dict(zip(best["model"], best["variant"]))

## 7. Caminho C2: alinhamento geométrico

**A ideia.** O fator A pergunta se o conteúdo virtual está encostado no objeto real, "sem uma separação
significativa". Em vez de perguntar ao VLM, medimos a separação em pixels. O VLM não é chamado de novo:
usamos o nome do objeto que ele já deu e as respostas dele sobre S e I.

**Passo a passo.**
1. **Conteúdo virtual (m_c):** diferença de pixels entre I_a e I_r (os pares são alinhados pixel a
   pixel).
2. **Objeto-chave (m_k):** o nome do objeto dado pelo VLM vai para o Grounding DINO sobre I_r, e a caixa
   de maior escore é segmentada pelo SAM.
3. **Medida `gap_norm`:** menor distância entre o conteúdo e o objeto, dividida pelo tamanho do
   conteúdo. Vale 0 quando eles se tocam.
4. **A geométrico:** A = 1 quando `gap_norm` fica abaixo de um limiar, escolhido por validação cruzada
   (5 partições): cada par é decidido por um limiar ajustado nos outros pares.
5. **Decisão híbrida:** M = A_geo ∧ S_vlm ∧ I_vlm. Onde o objeto não é detectado, vale o A do VLM.

Rodamos o C2 no prompt do artigo e na melhor variante de cada modelo.

In [ ]:
alignment_keys = sorted({(m, REFERENCE) for m in MODELS} | set(BEST.items()))
for model, variant in alignment_keys:
    run_dir = OUT_ROOT / "manipulation" / model / variant
    if not (run_dir / "metrics.json").exists() or (run_dir / "alignment" / "metrics.json").exists():
        continue
    if RUN_ALIGNMENT:
        print("alinhamento:", run_dir)
        run_alignment(AlignmentConfig(run_dir=str(run_dir), manifest=MANIFEST))

align = reporting.alignment_summary(OUT_ROOT, alignment_keys)

### 7.1 O C2 melhorou a decisão?

**Esta é a tabela que responde à pergunta.** Cada linha compara, na mesma execução, a decisão só com o
VLM e a decisão híbrida (com o A geométrico):

- **acurácia, precisão, recall, FPR, FNR:** "VLM → híbrido";
- **só híbrido acerta / só VLM acerta:** pares em que só uma das duas decisões está certa;
- **ganho?:** "melhorou" quando a acurácia híbrida é maior com McNemar p < 0,05.

In [ ]:
if align.empty:
    print("nenhuma execução com alinhamento ainda (rode a célula anterior com RUN_ALIGNMENT = True)")
else:
    cols = ["model", "variant"] + [c for c in align.columns if "→" in c] + ["só híbrido acerta", "só VLM acerta",
                                                                           "McNemar p", "ganho?"]
    display(align[cols])

### 7.2 Por que o C2 ajuda ou não: diagnóstico

O C2 só pode melhorar a decisão se duas coisas forem verdade: a medida geométrica julga A melhor que o
VLM, **e** as respostas do VLM sobre S e I são boas o suficiente. Como a decisão é um E, um "não"
errado em S ou I bloqueia o ataque, seja qual for o A.

- **detected:** em quantos pares o objeto citado pelo VLM foi encontrado.
- **AUC gap_norm:** probabilidade de um par desalinhado (A = 0) ter `gap_norm` maior que um par alinhado
  sorteado ao acaso. 0,5 = a medida não separa os dois grupos; 1 = separa perfeitamente.
- **acerto em A: geométrico × VLM:** fração dos pares (com objeto detectado) em que cada um acerta o
  rótulo de A. É a comparação direta entre a medida e o julgamento do VLM.
- **acurácia híbrida (média ± dp, CV):** a validação cruzada foi repetida 20 vezes com divisões
  diferentes; o desvio mostra o quanto o resultado depende da divisão.

In [ ]:
if not align.empty:
    diag = ["model", "variant", "detected"] + [c for c in align.columns if c.startswith("AUC")] + [
        "acerto em A: geométrico", "acerto em A: VLM (mesmos pares)", "acurácia híbrida (média ± dp, CV)"]
    display(align[diag])

O gráfico mostra a medida de uma execução: quanto mais os pontos laranja (A = 0) ficam à direita dos
azuis (A = 1), melhor a separação.

In [ ]:
EX_RUN = ("qwen3-vl-4b", REFERENCE)
feat_path = OUT_ROOT / "manipulation" / EX_RUN[0] / EX_RUN[1] / "alignment" / "features.csv"
if feat_path.exists() and EX_RUN in runs:
    feats = pd.read_csv(feat_path, dtype={"id": str}).merge(runs[EX_RUN][["id", "A"]], on="id")
    fig, table = viz.gap_strip(feats, "gap_norm", f"gap_norm por rótulo A ({EX_RUN[0]} / {EX_RUN[1]})")
    plt.show()
    display(table)

Pontos cheios: decisão só com o VLM; pontos vazados: a mesma execução com o A geométrico. Um ganho real
aproxima o ponto do canto (0, 0).

In [ ]:
for model in MODELS if not align.empty else []:
    t = align[align["model"] == model][["variant", "fpr", "fnr", "hybrid_fpr", "hybrid_fnr"]]
    if t.empty:
        continue
    fig, table = viz.operating_points(t, f"{model}: VLM (cheio) × híbrido com A geométrico (vazado)")
    plt.show()
    display(table)

### 7.3 Exemplos

O par aparece com o objeto detectado (verde) e o conteúdo virtual (laranja). `EX_FILTER` seleciona, por
exemplo, pares desalinhados (A = 0) que o VLM aceitou e o A geométrico corrigiu.

In [ ]:
EX_MODEL, EX_VARIANT, N_EXAMPLES = "llava-next-8b-4bit", REFERENCE, 2
pred_path = OUT_ROOT / "manipulation" / EX_MODEL / EX_VARIANT / "alignment" / "predictions.csv"
if pred_path.exists():
    h = pd.read_csv(pred_path, dtype={"id": str})
    h = h.merge(pd.read_csv(MANIFEST, dtype={"id": str})[["id", "raw_path", "ar_path"]], on="id")
    h[["raw_path", "ar_path"]] = h[["raw_path", "ar_path"]].map(lambda p: str((Path(MANIFEST).parent / p).resolve()))
    EX_FILTER = (h["A"] == 0) & (h["pred_A_vlm"] == 1) & (h["pred_A"] == 0)  # A_geo corrigiu o VLM
    print(f"{EX_FILTER.sum()} pares no filtro")
    for _, row in h[EX_FILTER].head(N_EXAMPLES).iterrows():
        mask = pred_path.parent / "key_masks" / f"{row['id']}.png"
        viz.show_alignment(row, str(mask) if mask.exists() else None)
        plt.show()

## 8. Resumo

Uma linha por execução: as métricas do artigo, o ganho em relação ao prompt `paper` e, quando existe, o
resultado do C2. Antes de concluir:

- leia algumas respostas brutas das variantes novas (seção 5.7);
- confira que não há respostas ilegíveis em excesso (linha "ilegível" da seção 5.3);
- lembre que as conclusões sobre I se apoiam em só 11 pares benignos.

In [ ]:
summary = gain.copy()
if len(align):
    c2 = align[["model", "variant", "acurácia: VLM → híbrido", "ganho?"]].rename(
        columns={"model": "modelo", "variant": "variante", "acurácia: VLM → híbrido": "C2: acurácia VLM → híbrido",
                 "ganho?": "C2: ganho?"})
    summary = summary.merge(c2, on=["modelo", "variante"], how="left")
display(summary)